# 패키지 및 데이터

In [6]:
# ===============================
# 기본 라이브러리
# ===============================
import numpy as np
import seaborn as sb
from matplotlib import pyplot as plt

from pandas import DataFrame, concat
from hossam import *


# ===============================
# scikit-learn: 파이프라인 & 전처리
# ===============================
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    learning_curve
)


# ===============================
# 로지스틱 회귀 모델
# ===============================
from sklearn.linear_model import LogisticRegression


# ===============================
# 성능 평가 지표
# ===============================
from sklearn.metrics import (
    log_loss,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_curve,
    roc_auc_score
)


### 데이터

In [1]:
origin = load_data("pima_indians_diabetes")
origin.head()

이 데이터 세트는 원래 미국 국립 당뇨병·소화기·신장질환 연구소(National Institute of Diabetes and Digestive and Kidney Diseases)에서 제공한 것입니다. 이 데이터 세트의 목적은 데이터 세트에 포함된 특정 진단 측정값을 기반으로 환자의 당뇨병 여부를 진단적으로 예측하는 것입니다. 더 큰 데이터베이스에서 이러한 사례를 선택하는 데에는 몇 가지 제약 조건이 적용되었습니다. 특히, 여기에 포함된 모든 환자는 21세 이상의 피마 인디언 혈통을 가진 여성입니다. (출처: https://www.kaggle.com/datasets/uciml/pima-indians-diabetes-database)

field                     type    description
------------------------  ------  --------------------------------
Pregnancies               연속형  임신횟수
Glucose                   연속형  포도당 부하 검사 수치
BloodPressure             연속형  혈압
SkinThickness             연속형  팔 삼두근 뒤쪽의 피하지방 측정값
Insulin                   연속형  혈청 인슐린
BMI                       연속형  체질량 지수
DiabetesPedigreeFunction  연속형  당뇨 내력 가중치 값
Age                       연속형  나이
Outcome                   명목형  당뇨여부(0 또는 1)



,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.600,0.627,50,1
1,1,85,66,29,0,26.600,0.351,31,0
2,8,183,64,0,0,23.300,0.672,32,1
3,1,89,66,23,94,28.100,0.167,21,0
4,0,137,40,35,168,43.100,2.288,33,1


# 데이터 품질 확인

### 명목형 타입 변환

In [3]:
df1 = origin.astype({"Outcome":"category"})
df1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype   
---  ------                    --------------  -----   
 0   Pregnancies               768 non-null    int64   
 1   Glucose                   768 non-null    int64   
 2   BloodPressure             768 non-null    int64   
 3   SkinThickness             768 non-null    int64   
 4   Insulin                   768 non-null    int64   
 5   BMI                       768 non-null    float64 
 6   DiabetesPedigreeFunction  768 non-null    float64 
 7   Age                       768 non-null    int64   
 8   Outcome                   768 non-null    category
dtypes: category(1), float64(2), int64(6)
memory usage: 49.0 KB


### 연속형 데이터의 품질 확인

In [4]:
def hs_describe(data, columns=None):
    """
    수치형 변수 요약 통계 + 결측/이상치/왜도 기반 분포 해석 함수
    """

    # ── 수치형 컬럼 자동 추출
    num_columns = list(data.select_dtypes(include=np.number).columns)

    if not columns:
        columns = num_columns

    # ── 기본 기술통계
    desc = data[columns].describe().T

    # ── 결측치 개수 / 비율
    na_counts = data[columns].isnull().sum()
    desc.insert(1, "na_count", na_counts)
    desc.insert(2, "na_rate", (na_counts / len(data)) * 100)

    # ── 추가 통계 저장용
    additional_stats = []

    for f in columns:
        # 수치형 변수만 처리
        if f not in num_columns:
            continue

        # ── 사분위수
        q1 = data[f].quantile(0.25)
        q3 = data[f].quantile(0.75)
        iqr = q3 - q1

        # ── 이상치 기준 (Tukey’s fences)
        down = q1 - 1.5 * iqr
        up = q3 + 1.5 * iqr

        # ── 왜도
        skew = data[f].skew()
        abs_skew = abs(skew)

        # ── 이상치 개수 / 비율
        outlier_count = ((data[f] < down) | (data[f] > up)).sum()
        outlier_rate = (outlier_count / len(data)) * 100

        # ── 분포 형태 해석
        if abs_skew < 0.5:
            dist = "거의 대칭"
        elif abs_skew < 1.0:
            dist = "약한 우측 꼬리" if skew > 0 else "약한 좌측 꼬리"
        elif abs_skew < 2.0:
            dist = "중간 우측 꼬리" if skew > 0 else "중간 좌측 꼬리"
        else:
            dist = "극단 우측 꼬리" if skew > 0 else "극단 좌측 꼬리"

        # ── 로그 변환 필요성 판단
        if abs_skew < 0.5:
            log_need = "낮음"
        elif abs_skew < 1.0:
            log_need = "중간"
        else:
            log_need = "높음"

        additional_stats.append({
            "field": f,
            "iqr": iqr,
            "down": down,
            "up": up,
            "outlier_count": outlier_count,
            "outlier_rate": outlier_rate,
            "skew": skew,
            "dist": dist,
            "log_need": log_need,
        })

    # ── 추가 통계 DataFrame
    additional_df = DataFrame(additional_stats).set_index("field")

    # ── 결과 병합
    result = concat([desc, additional_df], axis=1)

    return result

In [7]:
hs_describe(df1)

,count,na_count,na_rate,mean,std,min,25%,50%,75%,max,iqr,down,up,outlier_count,outlier_rate,skew,dist,log_need
Pregnancies,768.000,0,0.000,3.845,3.370,0.000,1.000,3.000,6.000,17.000,5.000,-6.500,13.500,4,0.521,0.902,약한 우측 꼬리,중간
Glucose,768.000,0,0.000,120.895,31.973,0.000,99.000,117.000,140.250,199.000,41.250,37.125,202.125,5,0.651,0.174,거의 대칭,낮음
BloodPressure,768.000,0,0.000,69.105,19.356,0.000,62.000,72.000,80.000,122.000,18.000,35.000,107.000,45,5.859,-1.844,중간 좌측 꼬리,높음
SkinThickness,768.000,0,0.000,20.536,15.952,0.000,0.000,23.000,32.000,99.000,32.000,-48.000,80.000,1,0.130,0.109,거의 대칭,낮음
Insulin,768.000,0,0.000,79.799,115.244,0.000,0.000,30.500,127.250,846.000,127.250,-190.875,318.125,34,4.427,2.272,극단 우측 꼬리,높음
BMI,768.000,0,0.000,31.993,7.884,0.000,27.300,32.000,36.600,67.100,9.300,13.350,50.550,19,2.474,-0.429,거의 대칭,낮음
DiabetesPedigreeFunction,768.000,0,0.000,0.472,0.331,0.078,0.244,0.372,0.626,2.420,0.382,-0.330,1.200,29,3.776,1.920,중간 우측 꼬리,높음
Age,768.000,0,0.000,33.241,11.760,21.000,24.000,29.000,41.000,81.000,17.000,-1.500,66.500,9,1.172,1.130,중간 우측 꼬리,높음


### 명목형 데이터의 품질 확인

In [10]:
def category_describe(data, columns=None):
    # 수치형 컬럼 목록
    num_columns = data.select_dtypes(include=np.number).columns

    # columns 미지정 시 → 범주형(object, category, bool) 자동 선택
    if not columns:
        columns = data.select_dtypes(
            include=["object", "category", "bool"]
        ).columns

    result = []
    summary = []

    for f in columns:
        # 숫자형 컬럼은 건너뜀
        if f in num_columns:
            continue

        # 범주별 빈도 (NaN 포함)
        value_counts = data[f].value_counts(dropna=False)

        # 컬럼에 값이 하나도 없으면 skip
        if len(value_counts) == 0:
            continue

        # ─────────────────────────────
        # 범주별 상세 테이블
        # ─────────────────────────────
        for category, count in value_counts.items():
            rate = (count / len(data)) * 100
            result.append({
                "변수": f,
                "범주": category,
                "빈도": count,
                "비율(%)": round(rate, 2),
            })

        # ─────────────────────────────
        # 최대 / 최소 범주 요약
        # ─────────────────────────────
        max_category = value_counts.index[0]  
        max_count = value_counts.iloc[0]   
        max_rate = (max_count / len(data)) * 100

        min_category = value_counts.index[-1] #맨 마지막 행에 접근?
        min_count = value_counts.iloc[-1]  
        min_rate = (min_count / len(data)) * 100

        #d는 d의 값끼리, j는 j의 값끼리

        summary.append({
            "변수": f,
            "최대_범주": max_category,
            "최대_비율(%)": round(max_rate, 2),
            "최소_범주": min_category,
            "최소_비율(%)": round(min_rate, 2),
        })

    return (
        DataFrame(result),
        DataFrame(summary).set_index("변수"),
    )


In [16]:
a, b =category_describe(df1)
display(a)
display(b)

,변수,범주,빈도,비율(%)
0,Outcome,0,500,65.100
1,Outcome,1,268,34.900


,최대_범주,최대_비율(%),최소_범주,최소_비율(%)
변수,,,,
Outcome,0,65.100,1,34.900


# 데이터전처리

### 결측치 정제

In [17]:
# 0값을 검사할 피처명 리스트
zero_features = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]

df2 = df1.copy()

df2[zero_features] = df2[zero_features].replace(0, np.nan)
df2.head()


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148.000,72.000,35.000,NaN,33.600,0.627,50,1
1,1,85.000,66.000,29.000,NaN,26.600,0.351,31,0
2,8,183.000,64.000,NaN,NaN,23.300,0.672,32,1
3,1,89.000,66.000,23.000,94.000,28.100,0.167,21,0
4,0,137.000,40.000,35.000,168.000,43.100,2.288,33,1


In [18]:
imr = SimpleImputer(missing_values=np.nan , strategy='median')
df_imr = imr.fit_transform(df2.values)

df3 = DataFrame(df_imr,
                index = df2.index,
                columns=df2.columns
                )

df3 = df3.astype({"Outcome":"category"})

desc = hs_describe(df3)
desc


,count,na_count,na_rate,mean,std,min,25%,50%,75%,max,iqr,down,up,outlier_count,outlier_rate,skew,dist,log_need
Pregnancies,768.000,0,0.000,3.845,3.370,0.000,1.000,3.000,6.000,17.000,5.000,-6.500,13.500,4,0.521,0.902,약한 우측 꼬리,중간
Glucose,768.000,0,0.000,121.656,30.438,44.000,99.750,117.000,140.250,199.000,40.500,39.000,201.000,0,0.000,0.536,약한 우측 꼬리,중간
BloodPressure,768.000,0,0.000,72.387,12.097,24.000,64.000,72.000,80.000,122.000,16.000,40.000,104.000,14,1.823,0.142,거의 대칭,낮음
SkinThickness,768.000,0,0.000,29.108,8.791,7.000,25.000,29.000,32.000,99.000,7.000,14.500,42.500,87,11.328,0.838,약한 우측 꼬리,중간
Insulin,768.000,0,0.000,140.672,86.383,14.000,121.500,125.000,127.250,846.000,5.750,112.875,135.875,346,45.052,3.380,극단 우측 꼬리,높음
BMI,768.000,0,0.000,32.455,6.875,18.200,27.500,32.300,36.600,67.100,9.100,13.850,50.250,8,1.042,0.599,약한 우측 꼬리,중간
DiabetesPedigreeFunction,768.000,0,0.000,0.472,0.331,0.078,0.244,0.372,0.626,2.420,0.382,-0.330,1.200,29,3.776,1.920,중간 우측 꼬리,높음
Age,768.000,0,0.000,33.241,11.760,21.000,24.000,29.000,41.000,81.000,17.000,-1.500,66.500,9,1.172,1.130,중간 우측 꼬리,높음


### 이상치 처리(로그변환)

In [46]:
log_fields = desc[desc['log_need'] != '낮음'].index.tolist()
# desc: 변수 요약 테이블
# != '낮음' -> 로그 변환이 필요한 변수만 선택
# log_fields: 해당 변수들의 칼럼별 리스트 싱성

df4 = df3.copy()

for f in log_fields:
    df4[f] = np.log1p(df4[f]) #log(1+x) 변환 적용

hs_describe(df4)

,count,na_count,na_rate,mean,std,min,25%,50%,75%,max,iqr,down,up,outlier_count,outlier_rate,skew,dist,log_need
Pregnancies,768.000,0,0.000,1.311,0.770,0.000,0.693,1.386,1.946,2.890,1.253,-1.186,3.825,0,0.000,-0.243,거의 대칭,낮음
Glucose,768.000,0,0.000,4.779,0.248,3.807,4.613,4.771,4.951,5.298,0.338,4.106,5.457,4,0.521,-0.058,거의 대칭,낮음
BloodPressure,768.000,0,0.000,72.387,12.097,24.000,64.000,72.000,80.000,122.000,16.000,40.000,104.000,14,1.823,0.142,거의 대칭,낮음
SkinThickness,768.000,0,0.000,3.359,0.314,2.079,3.258,3.401,3.497,4.605,0.238,2.900,3.854,96,12.500,-0.859,약한 좌측 꼬리,중간
Insulin,768.000,0,0.000,4.827,0.495,2.708,4.808,4.836,4.854,6.742,0.046,4.739,4.923,346,45.052,-0.175,거의 대칭,낮음
BMI,768.000,0,0.000,3.489,0.204,2.955,3.350,3.506,3.627,4.221,0.277,2.934,4.043,3,0.391,-0.038,거의 대칭,낮음
DiabetesPedigreeFunction,768.000,0,0.000,0.365,0.199,0.075,0.218,0.317,0.486,1.230,0.268,-0.184,0.888,13,1.693,1.118,중간 우측 꼬리,높음
Age,768.000,0,0.000,3.482,0.313,3.091,3.219,3.401,3.738,4.407,0.519,2.441,4.516,0,0.000,0.615,약한 우측 꼬리,중간


### 훈련, 검증 데이터 분리

In [47]:
df = df4

# 중요!!! 종속변수를 정수형으로 변환해야 한다.
df['Outcome'] = df['Outcome'].astype('int')
yname = "Outcome"
X = df.drop(columns=[yname])
y = df[yname]

X_train, X_test, y_train, y_test = train_test_split(X,y, test_size = 0.2, random_state=52)
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((614, 8), (154, 8), (614,), (154,))

# 로지스틱 모형

In [48]:
%%time

# ===============================
# Pipeline 정의
# ===============================
pipe = Pipeline(
    steps=[
        ("vif_selector", VIFSelector()),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(random_state=52)),
    ]
)


# ===============================
# GridSearch 파라미터
# ===============================
param_grid = {
    "model__penalty": ["l2"],
    "model__solver": ["lbfgs"],
    "model__C": [0.01, 0.1, 1, 10, 100],
    "model__max_iter": [100, 300, 500],
    "model__class_weight": [None, "balanced"],
}


# ===============================
# GridSearchCV
# ===============================
gs = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    cv=5,
    scoring="roc_auc",
    n_jobs=-1,
)


# ===============================
# 학습 및 결과
# ===============================
gs.fit(X_train, y_train)

best_estimator = gs.best_estimator_
best_estimator


CPU times: total: 172 ms
Wall time: 335 ms


Pipeline(steps=[('vif_selector', VIFSelector()), ('scaler', StandardScaler()),
                ('model',
                 LogisticRegression(C=1, class_weight='balanced',
                                    random_state=52))])

# 성능평가 지표

### 의사결정계수

In [49]:
r2 = estimator.score(X_test, y_test)
r2

NameError: name 'estimator' is not defined

### 로그 손실값

In [41]:
log_loss_test = -log_loss(y_test, y_pred_proba, normalize = False)
log_loss_test

NameError: name 'y_pred_proba' is not defined

### 로그손실 계산

In [37]:
y_null = np.ones_like(y_test) * y_test.mean()
log_loss_null = -log_los(y_test, y_null, normalize=False)
log_loss_null

NameError: name 'log_los' is not defined